# SciFact Midway Experiments

This notebook reproduces the midway-stage experiments for the project **Retrieval-Based Scientific Claim Verification with SciFact**. It compares a lexical baseline against a stronger retrieval + encoder pipeline, and it is designed to run either on CPU for smoke tests or on GPU for the real midway experiments.

## Experiment Setup

- Dataset: SciFact (`corpus.jsonl`, `claims_train.jsonl`, `claims_dev.jsonl`)
- Baseline retriever: TF-IDF cosine retrieval
- Stronger retriever presets: `all-MiniLM-L6-v2` for CPU quick runs, `all-mpnet-base-v2` for V100 runs
- Baseline verifier: TF-IDF + multinomial logistic regression
- Stronger verifier presets: lightweight BERT for CPU quick runs, `SciBERT` or `PubMedBERT` for GPU runs
- Main outputs: retrieval metrics, oracle document classification metrics, and end-to-end joint evidence hit

## Recommended Presets

- `cpu_quick`: sanity check on a local CPU
- `gpu_midway`: recommended for your actual midway report on a V100
- `gpu_strong`: optional stronger run if you have extra GPU time

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from scifact_midway import build_experiment_config, run_all_experiments

RESULTS_DIR = PROJECT_ROOT / 'results' / 'midway'
SUMMARY_PATH = RESULTS_DIR / 'summary.json'
PROJECT_ROOT

In [ ]:
PRESET = 'gpu_midway'
RUN_FULL_EXPERIMENT = False
RESULTS_DIR = PROJECT_ROOT / 'results' / PRESET
SUMMARY_PATH = RESULTS_DIR / 'summary.json'

if RUN_FULL_EXPERIMENT or not SUMMARY_PATH.exists():
    config = build_experiment_config(PRESET)
    results = run_all_experiments(project_root=PROJECT_ROOT, output_dir=RESULTS_DIR, config=config)
else:
    results = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

results.keys()

## Dataset Summary

In [ ]:
dataset_summary = results['dataset_summary']
pd.Series(dataset_summary)

## Retrieval Results

In [ ]:
retrieval_df = pd.read_csv(RESULTS_DIR / 'retrieval_metrics.csv', index_col=0)
retrieval_df.round(4)

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'plots' / 'retrieval_recall_curve.png')))

## Oracle Document Classification

In [ ]:
classification_df = pd.read_csv(RESULTS_DIR / 'classification_metrics.csv', index_col=0)
classification_df.round(4)

In [ ]:
display(Image(filename=str(RESULTS_DIR / 'plots' / 'logreg_confusion_matrix.png')))
display(Image(filename=str(RESULTS_DIR / 'plots' / 'bert_tiny_confusion_matrix.png')))

## End-to-End Pipeline Results

In [ ]:
pipeline_df = pd.read_csv(RESULTS_DIR / 'pipeline_metrics.csv', index_col=0)
pipeline_df.round(4)

## Quick Takeaways

In [ ]:
print('Dense retrieval should improve retrieval quality clearly over TF-IDF.')
print('A scientific encoder such as SciBERT should improve macro-F1 and especially the REFUTES class compared with logistic regression.')
print('Any remaining end-to-end gap is useful midway material because it leaves clear room for the final report.')

## Optional: Inspect Predictions

These files can be useful when writing the Results section or doing error analysis.

In [ ]:
bert_pipeline_preds = pd.read_csv(RESULTS_DIR / 'dense_plus_bert_pipeline_predictions.csv')
bert_pipeline_preds.head()